In [ ]:
import polars as pl

df = pl.read_csv("../../data/wallets.csv")

print(df)

shape: (2_573, 4)
┌─────────────────────────────────┬────────────┬───────────────┬─────────┐
│ address                         ┆ account_id ┆ asset         ┆ balance │
│ ---                             ┆ ---        ┆ ---           ┆ ---     │
│ str                             ┆ i64        ┆ str           ┆ f64     │
╞═════════════════════════════════╪════════════╪═══════════════╪═════════╡
│ THhsnT9tKokP42gKscXPmT7DB3BWTi… ┆ 15002      ┆ TRX_USDT_S2UZ ┆ 0.0     │
│ TL6awvGnCaNvf5MfueR7U2DDyCrdUj… ┆ 15003      ┆ TRX_USDT_S2UZ ┆ 0.0     │
│ TUS6FqQ2tx318CTQL5kcr9swvHQhdw… ┆ 15004      ┆ TRX_USDT_S2UZ ┆ 0.0     │
│ TCXLtS29eS7VsjyfcaQsQPBd9cGv16… ┆ 15005      ┆ TRX_USDT_S2UZ ┆ 0.0     │
│ TK2QR62hyvRxsaCcbTvMzpxU5WeSCf… ┆ 15006      ┆ TRX_USDT_S2UZ ┆ 0.0     │
│ …                               ┆ …          ┆ …             ┆ …       │
│ TM5YjPVuXLeYBynCpBQwPNZsJsqjvX… ┆ 18878      ┆ TRX_USDT_S2UZ ┆ 0.0     │
│ TSNRa1AmWnqSsrbAXYWvf3rpYfPyYu… ┆ 18879      ┆ TRX_USDT_S2UZ ┆ 0.0     │
│ TGPUQ

In [24]:
addresses = df["address"].to_list()

In [ ]:
import asyncio, aiohttp, time, random, csv

API_KEY = "api_key_here"
ADDRESSES = addresses

BASE_URL = "https://services.tokenview.io/vipapi/usdt/addressdetail"
MAX_CONCURRENCY = 50
RETRY_STATUS = {429, 500, 502, 503, 504}

class RateLimiter:
    def __init__(self, capacity: int, refill_per_sec: float):
        self.capacity = capacity
        self.tokens = capacity
        self.refill_per_sec = refill_per_sec
        self.last = time.monotonic()
        self._lock = asyncio.Lock()

    async def acquire(self):
        while True:
            async with self._lock:
                now = time.monotonic()
                elapsed = now - self.last
                self.last = now
                self.tokens = min(self.capacity, self.tokens + elapsed * self.refill_per_sec)
                if self.tokens >= 1:
                    self.tokens -= 1
                    return
            await asyncio.sleep(0.05)

RATE_LIMIT_PER_MIN = 300
rate_limiter = RateLimiter(capacity=RATE_LIMIT_PER_MIN, refill_per_sec=RATE_LIMIT_PER_MIN / 60.0)

async def fetch_balance(session, addr):
    await rate_limiter.acquire()
    url = f"{BASE_URL}/{addr}?apikey={API_KEY}"
    backoff = 1.0
    for attempt in range(6):
        try:
            async with session.get(url, timeout=20) as r:
                if r.status in RETRY_STATUS:
                    await asyncio.sleep(backoff + random.random() * 0.25)
                    backoff = min(backoff * 2, 16)
                    continue
                data = await r.json(content_type=None)
                if data.get("code") == 1:
                    bal_str = (data.get("data") or {}).get("balance", "0") or "0"
                    try:
                        bal = float(bal_str)
                    except (TypeError, ValueError):
                        bal = None
                    return addr, bal, None
                else:
                    return addr, None, data
        except asyncio.TimeoutError:
            await asyncio.sleep(backoff); backoff = min(backoff * 2, 16)
        except aiohttp.ClientError as e:
            await asyncio.sleep(backoff); backoff = min(backoff * 2, 16)
        except Exception as e:
            return addr, None, {"exception": repr(e)}
    return addr, None, {"error": "exhausted_retries"}

async def run(addresses):
    sem = asyncio.Semaphore(MAX_CONCURRENCY)
    async with aiohttp.ClientSession(headers={"Connection": "keep-alive"}) as session:
        async def bounded(addr):
            async with sem:
                return await fetch_balance(session, addr)
        tasks = [asyncio.create_task(bounded(a)) for a in addresses]
        results = []
        for coro in asyncio.as_completed(tasks):
            results.append(await coro)
        return results

def write_csv(results, filename="balances.csv"):
    with open(filename, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Address", "Balance", "Error"])
        for addr, bal, err in results:
            writer.writerow([addr, bal if bal is not None else "", err if err else ""])

results = await run(ADDRESSES)
write_csv(results)
print(f"Done. Wrote {len(results)} rows to balances.csv")

Done. Wrote 2573 rows to balances.csv


In [6]:
validations_df = pl.read_csv("../../balances.csv")

print(validations_df)

shape: (2_573, 3)
┌─────────────────────────────────┬─────────┬─────────────────────────────────┐
│ Address                         ┆ Balance ┆ Error                           │
│ ---                             ┆ ---     ┆ ---                             │
│ str                             ┆ f64     ┆ str                             │
╞═════════════════════════════════╪═════════╪═════════════════════════════════╡
│ TPyCSoygyhgQjmEnErexXSgk2scdQG… ┆ null    ┆ {'code': 404, 'msg': '无数据',  │
│                                 ┆         ┆ 'e…                             │
│ TF33cXZLKZtS1sVtj5rX1aXvDagdNX… ┆ null    ┆ {'code': 404, 'msg': '无数据',  │
│                                 ┆         ┆ 'e…                             │
│ TETK8xsEUvaNo7PgGGTvAxeLdCyKRD… ┆ null    ┆ {'code': 404, 'msg': '无数据',  │
│                                 ┆         ┆ 'e…                             │
│ TVhJM9ifRyH1dmk3sSdNQcuDFeptwK… ┆ null    ┆ {'code': 404, 'msg': '无数据',  │
│                                 

In [7]:
checker = validations_df.join(df, left_on="Address", right_on="address" ,  how="inner")

In [38]:
validations_df.count()

Address,Balance,Error
u32,u32,u32
2573,312,2261


In [39]:

df.count()

address,account_id,asset,balance
u32,u32,u32,u32
2573,2573,2573,2573


In [40]:
checker.count()

Address,Balance,Error,account_id,asset,balance
u32,u32,u32,u32,u32,u32
2573,312,2261,2573,2573,2573


In [8]:
print(checker)

shape: (2_573, 6)
┌────────────────────────┬─────────┬────────────────────────┬────────────┬───────────────┬─────────┐
│ Address                ┆ Balance ┆ Error                  ┆ account_id ┆ asset         ┆ balance │
│ ---                    ┆ ---     ┆ ---                    ┆ ---        ┆ ---           ┆ ---     │
│ str                    ┆ f64     ┆ str                    ┆ i64        ┆ str           ┆ f64     │
╞════════════════════════╪═════════╪════════════════════════╪════════════╪═══════════════╪═════════╡
│ THhsnT9tKokP42gKscXPmT ┆ null    ┆ {'code': 404, 'msg':   ┆ 15002      ┆ TRX_USDT_S2UZ ┆ 0.0     │
│ 7DB3BWTi…              ┆         ┆ '无数据', 'e…          ┆            ┆               ┆         │
│ TL6awvGnCaNvf5MfueR7U2 ┆ 0.0     ┆ null                   ┆ 15003      ┆ TRX_USDT_S2UZ ┆ 0.0     │
│ DDyCrdUj…              ┆         ┆                        ┆            ┆               ┆         │
│ TUS6FqQ2tx318CTQL5kcr9 ┆ null    ┆ {'code': 404, 'msg':   ┆ 15004      ┆ T

In [21]:
def test_balances():
    validated_df = checker.filter(pl.col("Balance") == pl.col("balance"))
    flagged_df = checker.filter(pl.col("Balance") != pl.col("balance"))
    null_df = checker.filter(pl.col("Balance").is_null() | pl.col("balance").is_null())
    
    validated_df.write_csv("../../data/validated.csv")
    flagged_df.write_csv("../../data/flagged.csv")
    null_df.write_csv("../../data/nulls.csv")

    return test_records(validated_df, flagged_df, null_df, checker.count())

In [22]:
def test_records(validated_df, flagged_df, null_df, expected_count):
    return validated_df.count() + flagged_df.count() + null_df.count() == expected_count


In [23]:
test_balances()

Address,Balance,Error,account_id,asset,balance
bool,bool,bool,bool,bool,bool
true,true,true,true,true,true


In [14]:
validated_df = checker.filter(pl.col("Balance") == pl.col("balance"))
flagged_df = checker.filter(pl.col("Balance") != pl.col("balance"))
null_df = checker.filter(pl.col("Balance").is_null() | pl.col("balance").is_null())


In [16]:
2261 + 5 + 307

2573

In [15]:
null_df.count()

Address,Balance,Error,account_id,asset,balance
u32,u32,u32,u32,u32,u32
2261,0,2261,2261,2261,2261


In [17]:
print(flagged_df)

shape: (5, 6)
┌─────────────────────────────────┬────────────┬───────┬────────────┬───────────────┬─────────┐
│ Address                         ┆ Balance    ┆ Error ┆ account_id ┆ asset         ┆ balance │
│ ---                             ┆ ---        ┆ ---   ┆ ---        ┆ ---           ┆ ---     │
│ str                             ┆ f64        ┆ str   ┆ i64        ┆ str           ┆ f64     │
╞═════════════════════════════════╪════════════╪═══════╪════════════╪═══════════════╪═════════╡
│ TRrXFC59jHEEPKhnfDewiBqyTKB4SE… ┆ 0.021      ┆ null  ┆ 16549      ┆ TRX_USDT_S2UZ ┆ 0.0     │
│ TTVJjisyU2tdYAZbTyYzfyZt34hpAs… ┆ 0.0        ┆ null  ┆ 16911      ┆ TRX_USDT_S2UZ ┆ 9.2     │
│ TU1fdToYjo3TFgUZUzNqgE8soxsLhr… ┆ 0.021201   ┆ null  ┆ 17854      ┆ TRX_USDT_S2UZ ┆ 0.0     │
│ TXo6ug52mUzbdWcapS5KGGxv4YGqA4… ┆ 0.021101   ┆ null  ┆ 18152      ┆ TRX_USDT_S2UZ ┆ 0.0     │
│ TBbcRSk5TqAvZFMGUGvxQ3pkQWKrb5… ┆ 157.457568 ┆ null  ┆ 18700      ┆ TRX_USDT_S2UZ ┆ 0.0     │
└─────────────────────────

In [13]:
flagged_df.count()

Address,Balance,Error,account_id,asset,balance
u32,u32,u32,u32,u32,u32
5,5,0,5,5,5


In [12]:
validated_df.count()

Address,Balance,Error,account_id,asset,balance
u32,u32,u32,u32,u32,u32
307,307,0,307,307,307


In [ ]:
import requests

API_KEY = "api_key_here"
WALLET = "TPoRaauoq3eYtkcEmkfdvof9MT9QeriP9n"
USDT_CONTRACT = "TXLAQ63Xg1NAzckPwKHvzw7CSEmLMEqcdj"

url = f"https://services.tokenview.io/vipapi/usdt/addressdetail/{WALLET}?apikey={API_KEY}"
resp = requests.get(url).json()

print(resp)

if resp.get("code") == 1:
    balance = float(resp["data"]["balance"])
    print("USDT Balance:", balance)
else:
    print("Error:", resp)

{'code': 1, 'msg': '成功', 'enMsg': 'SUCCESS', 'data': {'type': 'address', 'network': 'TRX', 'hash': 'TPoRaauoq3eYtkcEmkfdvof9MT9QeriP9n', 'txCount': 27, 'spend': '-50805.065', 'receive': '50805.097', 'balance': '0.032', 'txs': [{'type': 'tx', 'network': 'TRX', 'block_no': 46032298, 'height': 46032298, 'index': 53, 'time': 1668683355, 'txid': 'e18b9f7f428480daac77ab23e5b9d975caddcee0c4a2f39416c8e3ed26eeb302', 'confirmations': 30212501, 'from': 'TKJBPbSmZbDTvZGyVaaiVN6UqZh9iXLgo6', 'to': 'TPoRaauoq3eYtkcEmkfdvof9MT9QeriP9n', 'value': '1000', 'token': 'TR7NHqjeKQxGTCi8q8ZY4pL8otSzgjLj6t'}, {'type': 'tx', 'network': 'TRX', 'block_no': 46032265, 'height': 46032265, 'index': 345, 'time': 1668683256, 'txid': '7b14747535c8ca8aa147aa40a4667ad9e6db803b198ed27d97626d77ccbce410', 'confirmations': 30212534, 'from': 'TLwGzKjyxMUdxLRCQBtzy9ufEaV7TXLgo6', 'to': 'TPoRaauoq3eYtkcEmkfdvof9MT9QeriP9n', 'value': '10000', 'token': 'TR7NHqjeKQxGTCi8q8ZY4pL8otSzgjLj6t'}, {'type': 'tx', 'network': 'TRX', 'block